# 74 套数据 · 条件 CVAE · 生成与量化评估（MVP）

**推荐 Colab 文件**：本 notebook（`260523-74条件生成与量化评估（MVP）.ipynb`）

| 步骤 | 作用 |
|---|---|
| Step 0–4 | 安装依赖、配置、预处理、建模、训练 |
| **Step R** | 加载已训练权重（需先跑 Step 1 + 3） |
| Step 5–7 | 重建评估、条件生成、量化对比 |
| Step 8 | 交互式生成（ipywidgets） |


In [ ]:

# Step 0: 安装依赖（Colab 首次运行）
!pip -q install torch_geometric


In [ ]:

# Step 1: 全局配置 + 数据工具
import os, json, random, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader as GraphDataLoader
from torch_geometric.nn import HeteroConv, SAGEConv, GATv2Conv, global_mean_pool

USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        USE_DRIVE = False

def resolve_data_dir():
    candidates = [
        '/content/drive/MyDrive/master_thesis/data',
        '/content/drive/MyDrive/260509_dataset',
        '/content/data',
        os.path.join(os.getcwd(), 'data'),
        os.path.abspath(os.path.join(os.getcwd(), '..', 'data')),
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    return candidates[0]

DATA_DIR = resolve_data_dir()

OUT_DIR = os.path.join(DATA_DIR, 'processed_tensors_v2')
CACHE_VERSION = 'mvp_v2_lighting_cvae'

VOXEL_SIZE = 300.0
RES_X, RES_Y, RES_Z = 64, 128, 32

CHANNEL_MAP = {
    'empty': 0, 'entryway': 1, 'living_room': 2, 'dining_room': 3,
    'kitchen': 4, 'bedroom': 5, 'bathroom': 6, 'corridor': 7,
    'stairs': 8, 'utility': 9, 'balcony': 10, 'multi_purpose': 11,
}
ROOM_TYPES = [k for k in CHANNEL_MAP if k != 'empty']
NUM_CHANNELS = len(CHANNEL_MAP)
NODE_IN_DIM = 8
COND_DIM = 3 + len(ROOM_TYPES) + 3

LIGHTING_ACCESS_MAP = {'direct': 1.0, 'indirect': 0.5, 'none': 0.0}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')
print(f'数据目录: {DATA_DIR}  存在: {os.path.isdir(DATA_DIR)}')
print(f'节点维={NODE_IN_DIM}  条件维={COND_DIM}')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_node_supertype(room_type):
    if room_type in ['bedroom', 'living_room', 'dining_room', 'multi_purpose']:
        return 'living'
    if room_type in ['kitchen', 'bathroom', 'utility', 'balcony']:
        return 'service'
    return 'circulation'


def check_hetero_adjacency(r1, r2, tol=150, min_shared_len=300):
    if r1.get('floor', 1) != r2.get('floor', 1):
        if r1['type'] == 'stairs' or r2['type'] == 'stairs':
            ox = max(0, min(r1['box_max'][0], r2['box_max'][0]) - max(r1['box_min'][0], r2['box_min'][0]))
            oy = max(0, min(r1['box_max'][1], r2['box_max'][1]) - max(r1['box_min'][1], r2['box_min'][1]))
            if ox > 0 and oy > 0:
                return 'vertical'
        return None
    if r1.get('floor', 1) == r2.get('floor', 1):
        ox = max(0, min(r1['box_max'][0], r2['box_max'][0]) - max(r1['box_min'][0], r2['box_min'][0]) + tol)
        oy = max(0, min(r1['box_max'][1], r2['box_max'][1]) - max(r1['box_min'][1], r2['box_min'][1]) + tol)
        if (ox > min_shared_len and oy > 0) or (oy > min_shared_len and ox > 0):
            return 'horizontal'
    return None


def effective_lighting_ratio(room):
    dx = room['box_max'][0] - room['box_min'][0]
    dy = room['box_max'][1] - room['box_min'][1]
    floor_area = max(dx * dy, 1.0)
    eff = room.get('effective_lighting', [])
    total = sum(float(e.get('area', 0)) * float(e.get('attenuation', 1.0)) for e in eff)
    return min(total / floor_area, 5.0) / 5.0


def build_room_features(room, build_min, b_size):
    dx = room['box_max'][0] - room['box_min'][0]
    dy = room['box_max'][1] - room['box_min'][1]
    area = (dx * dy) / 1e6
    aspect = max(dx, dy) / max(min(dx, dy), 1.0)
    cx = ((room['box_min'][0] + room['box_max'][0]) / 2 - build_min[0]) / (b_size[0] + 1e-5)
    cy = ((room['box_min'][1] + room['box_max'][1]) / 2 - build_min[1]) / (b_size[1] + 1e-5)
    cz = ((room['box_min'][2] + room['box_max'][2]) / 2 - build_min[2]) / (b_size[2] + 1e-5)
    lighting_access = LIGHTING_ACCESS_MAP.get(room.get('lighting_access', 'none'), 0.0)
    lighting_priority = float(room.get('lighting_priority', 0)) / 10.0
    lighting_ratio = effective_lighting_ratio(room)
    return [area, aspect, cx, cy, cz, lighting_access, lighting_priority, lighting_ratio]


def build_condition_vector(data):
    meta = data.get('metadata', {})
    stats = meta.get('stats', {})
    bsize = meta.get('building_size', {'x': 1.0, 'y': 1.0, 'z': 1.0})
    rooms = data.get('rooms', [])
    total = max(len(rooms), 1)

    cond = [
        float(bsize.get('x', 1.0)) / 30000.0,
        float(bsize.get('y', 1.0)) / 30000.0,
        float(bsize.get('z', 1.0)) / 9000.0,
    ]
    for rt in ROOM_TYPES:
        cond.append(float(stats.get(rt, 0)) / total)

    direct = indirect = none = 0
    for r in rooms:
        acc = r.get('lighting_access', 'none')
        if acc == 'direct':
            direct += 1
        elif acc == 'indirect':
            indirect += 1
        else:
            none += 1
    cond.extend([direct / total, indirect / total, none / total])
    return cond


def json_to_sample(data):
    rooms = data.get('rooms', [])
    if not rooms:
        return None

    all_coords = np.array([r['box_min'] for r in rooms] + [r['box_max'] for r in rooms])
    build_min = all_coords.min(axis=0)
    build_max = all_coords.max(axis=0)
    b_size = build_max - build_min

    hetero = HeteroData()
    nodes_dict = {'living': [], 'service': [], 'circulation': []}
    id_to_idx = {}

    for r in rooms:
        ntype = get_node_supertype(r['type'])
        id_to_idx[r['id']] = (ntype, len(nodes_dict[ntype]))
        nodes_dict[ntype].append(build_room_features(r, build_min, b_size))

    for ntype, feats in nodes_dict.items():
        hetero[ntype].x = torch.tensor(feats, dtype=torch.float32) if feats else torch.empty((0, NODE_IN_DIM))

    edges_dict = {}
    for i in range(len(rooms)):
        for j in range(i + 1, len(rooms)):
            r1, r2 = rooms[i], rooms[j]
            rel = check_hetero_adjacency(r1, r2)
            if not rel:
                continue
            t1, idx1 = id_to_idx[r1['id']]
            t2, idx2 = id_to_idx[r2['id']]
            for src_t, dst_t, si, di in ((t1, t2, idx1, idx2), (t2, t1, idx2, idx1)):
                key = (src_t, rel, dst_t)
                edges_dict.setdefault(key, []).append([si, di])

    for key, elist in edges_dict.items():
        hetero[key].edge_index = torch.tensor(elist, dtype=torch.long).t().contiguous()

    grid = np.zeros((RES_X, RES_Y, RES_Z), dtype=np.int8)
    phys_center_xy = (build_min[:2] + build_max[:2]) / 2.0
    offset_xy = np.array([RES_X * VOXEL_SIZE / 2, RES_Y * VOXEL_SIZE / 2]) - phys_center_xy
    z_min_phys = build_min[2]

    for r in rooms:
        ch = CHANNEL_MAP.get(r['type'], 0)
        ix_min = int((r['box_min'][0] + offset_xy[0]) / VOXEL_SIZE)
        ix_max = int((r['box_max'][0] + offset_xy[0]) / VOXEL_SIZE)
        iy_min = int((r['box_min'][1] + offset_xy[1]) / VOXEL_SIZE)
        iy_max = int((r['box_max'][1] + offset_xy[1]) / VOXEL_SIZE)
        iz_min = int((r['box_min'][2] - z_min_phys) / VOXEL_SIZE)
        iz_max = int((r['box_max'][2] - z_min_phys) / VOXEL_SIZE)
        grid[max(0, ix_min):min(RES_X, ix_max), max(0, iy_min):min(RES_Y, iy_max), max(0, iz_min):min(RES_Z, iz_max)] = ch

    one_hot = np.zeros((NUM_CHANNELS, RES_X, RES_Y, RES_Z), dtype=np.float32)
    for c in range(NUM_CHANNELS):
        one_hot[c] = (grid == c).astype(np.float32)

    cond = torch.tensor(build_condition_vector(data), dtype=torch.float32)
    return {'graph': hetero, 'voxel': torch.tensor(one_hot, dtype=torch.float32), 'condition': cond}


def dice_loss_with_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (0, 2, 3, 4)
    inter = (probs * targets).sum(dims)
    union = probs.sum(dims) + targets.sum(dims)
    dice = (2 * inter + eps) / (union + eps)
    return 1.0 - dice.mean()


def graph_batch_size(batch):
    return int(getattr(batch, 'num_graphs', 1))


def graph_batch_dict(batch):
    if hasattr(batch, 'batch_dict'):
        return batch.batch_dict
    bd = {}
    for ntype in batch.node_types:
        if batch[ntype].x.size(0) > 0:
            if hasattr(batch[ntype], 'batch'):
                bd[ntype] = batch[ntype].batch
            else:
                bd[ntype] = torch.zeros(
                    batch[ntype].x.size(0), dtype=torch.long, device=batch[ntype].x.device
                )
    return bd


def graph_condition(batch):
    bs = graph_batch_size(batch)
    if not hasattr(batch, 'condition'):
        raise AttributeError('batch 缺少 condition：请用 prepare_graph_batch(hg, condition=tensor)')
    cond = batch.condition
    if cond.dim() == 1:
        cond = cond.unsqueeze(0)
    return cond.view(bs, -1)


def prepare_graph_batch(hg, condition=None):
    """将 condition 挂到 HeteroData 并确保 Batch 后仍可读取。"""
    if condition is not None:
        cond = condition.unsqueeze(0) if condition.dim() == 1 else condition
        hg.condition = cond
    batch = next(iter(GraphDataLoader([hg], batch_size=1)))
    if condition is not None and not hasattr(batch, 'condition'):
        batch.condition = hg.condition
    return batch


def forward_model(model, batch, condition=None):
    """显式传入 condition，避免 HeteroDataBatch 丢字段。"""
    cond = condition if condition is not None else graph_condition(batch)
    if cond.dim() == 1:
        cond = cond.unsqueeze(0)
    mu, logvar = model.encoder(
        batch.x_dict, batch.edge_index_dict, graph_batch_dict(batch)
    )
    z = model.reparameterize(mu, logvar)
    logits = model.decoder(z, cond.to(z.device))
    return logits, mu, logvar

from datetime import datetime

TOTAL_JSON_COUNT = len([
    f for f in glob.glob(os.path.join(DATA_DIR, 'house_*.json'))
    if not f.endswith('_topology.json')
])
TRAIN_SAMPLE_COUNT = None
TRAIN_TIMESTAMP = None
EXPERIMENT_META = {}


def set_experiment_meta(train_count, total_count=None, timestamp=None, extra=None):
    global TRAIN_SAMPLE_COUNT, TRAIN_TIMESTAMP, EXPERIMENT_META, TOTAL_JSON_COUNT
    TRAIN_SAMPLE_COUNT = int(train_count)
    if total_count is not None:
        TOTAL_JSON_COUNT = int(total_count)
    TRAIN_TIMESTAMP = timestamp or datetime.now().strftime('%Y%m%d_%H%M%S')
    EXPERIMENT_META = {
        'train_samples': TRAIN_SAMPLE_COUNT,
        'total_json': TOTAL_JSON_COUNT,
        'trained_at': TRAIN_TIMESTAMP,
    }
    if extra:
        EXPERIMENT_META.update(extra)
    return EXPERIMENT_META


def experiment_report_name(prefix, ext='csv'):
    ts = TRAIN_TIMESTAMP or datetime.now().strftime('%Y%m%d_%H%M%S')
    n_train = TRAIN_SAMPLE_COUNT if TRAIN_SAMPLE_COUNT is not None else 'NA'
    n_total = TOTAL_JSON_COUNT if TOTAL_JSON_COUNT is not None else 'NA'
    return f"{prefix}_json{n_total}_train{n_train}_{ts}.{ext}"


def experiment_report_path(prefix, subdir='eval_reports', ext='csv'):
    out = os.path.join(DATA_DIR, subdir)
    os.makedirs(out, exist_ok=True)
    return os.path.join(out, experiment_report_name(prefix, ext))




In [ ]:

# Step 2: 批量预处理 JSON -> .pt（含缓存版本号）
os.makedirs(OUT_DIR, exist_ok=True)
meta_path = os.path.join(OUT_DIR, '_cache_meta.json')

json_files = sorted([
    f for f in glob.glob(os.path.join(DATA_DIR, 'house_*.json'))
    if not f.endswith('_topology.json')
])
if not json_files:
    raise FileNotFoundError(f'未找到 JSON: {DATA_DIR}')

need_rebuild = True
if os.path.exists(meta_path):
    with open(meta_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    need_rebuild = meta.get('version') != CACHE_VERSION or meta.get('count') != len(json_files)

if need_rebuild:
    ok, fail = 0, 0
    for fp in json_files:
        fname = os.path.basename(fp)
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)
            sample = json_to_sample(data)
            if sample is None:
                continue
            torch.save(sample, os.path.join(OUT_DIR, fname.replace('.json', '.pt')))
            ok += 1
        except Exception as exc:
            fail += 1
            print(f'[FAIL] {fname}: {exc}')
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({'version': CACHE_VERSION, 'count': len(json_files), 'ok': ok, 'fail': fail}, f, indent=2)
    print(f'预处理完成: 成功 {ok}, 失败 {fail}, 输出 {OUT_DIR}')
else:
    print(f'命中缓存 {CACHE_VERSION}，跳过预处理（共 {len(json_files)} 个 JSON）')


In [ ]:

# Step 3: CVAE 模型（GNN 编码 + 条件注入 3D 解码）
LATENT_DIM = 256
HIDDEN_DIM = 128


def build_hetero_convs(in_dim, out_dim, vertical_gat=False):
    h_conv = {}
    node_types = ['living', 'service', 'circulation']
    for s in node_types:
        for d in node_types:
            h_conv[(s, 'horizontal', d)] = SAGEConv(in_dim, out_dim)
    vertical_pairs = [
        ('circulation', 'living'), ('living', 'circulation'),
        ('circulation', 'service'), ('service', 'circulation'),
        ('circulation', 'circulation'),
    ]
    for s, d in vertical_pairs:
        if vertical_gat:
            h_conv[(s, 'vertical', d)] = GATv2Conv(
                in_dim, out_dim, heads=2, concat=False, add_self_loops=False
            )
        else:
            h_conv[(s, 'vertical', d)] = SAGEConv(in_dim, out_dim)
    return HeteroConv(h_conv, aggr='sum')


class HeteroGraphVAEEncoder(nn.Module):
    def __init__(self, node_in_dim=NODE_IN_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv1 = build_hetero_convs(node_in_dim, hidden_dim, vertical_gat=True)
        self.conv2 = build_hetero_convs(hidden_dim, hidden_dim, vertical_gat=False)
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)

    def _pool(self, x_dict, batch_dict):
        feats = []
        for ntype, x in x_dict.items():
            if ntype in batch_dict and x.size(0) > 0:
                feats.append(global_mean_pool(x, batch_dict[ntype]))
        if not feats:
            dev = x_dict[list(x_dict.keys())[0]].device
            return torch.zeros(1, HIDDEN_DIM, device=dev)
        return torch.stack(feats, dim=0).mean(dim=0)

    def forward(self, x_dict, edge_index_dict, batch_dict):
        x_dict = {k: torch.relu(self.conv1(x_dict, edge_index_dict)[k]) for k in x_dict}
        x_dict = {k: torch.relu(self.conv2(x_dict, edge_index_dict)[k]) for k in x_dict}
        h = self._pool(x_dict, batch_dict)
        return self.to_mu(h), self.to_logvar(h)


class ConditionalVoxelDecoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, cond_dim=COND_DIM, channels=NUM_CHANNELS):
        super().__init__()
        self.init_volume_size = (256, 4, 8, 2)
        in_dim = latent_dim + cond_dim
        self.fc = nn.Linear(in_dim, int(np.prod(self.init_volume_size)))
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(256, 128, 4, stride=2, padding=1), nn.BatchNorm3d(128), nn.ReLU(True),
            nn.ConvTranspose3d(128, 64, 4, stride=2, padding=1), nn.BatchNorm3d(64), nn.ReLU(True),
            nn.ConvTranspose3d(64, 32, 4, stride=2, padding=1), nn.BatchNorm3d(32), nn.ReLU(True),
            nn.ConvTranspose3d(32, channels, 4, stride=2, padding=1),
        )

    def forward(self, z, cond):
        x = torch.cat([z, cond], dim=-1)
        d = self.fc(x).view(z.size(0), *self.init_volume_size)
        return self.decoder(d)


class SpatialModalCVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = HeteroGraphVAEEncoder()
        self.decoder = ConditionalVoxelDecoder()

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, batch):
        mu, logvar = self.encoder(
            batch.x_dict, batch.edge_index_dict, graph_batch_dict(batch)
        )
        z = self.reparameterize(mu, logvar)
        cond = graph_condition(batch)
        logits = self.decoder(z, cond)
        return logits, mu, logvar


model = SpatialModalCVAE().to(device)
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')


## Step R: 快速恢复（推荐）

**Colab 重启后内存会清空**，Step R **不能作为第一个 cell 单独运行**（否则会 `NameError: os` 或缺少 `SpatialModalCVAE`）。

| 场景 | 需要运行 |
|---|---|
| 首次完整训练 | Step 0 → 1 → 2 → 3 → 4 |
| **Colab 重启 / 断线后** | Step 0（若需）→ **1 → 3 → R** → Step 6/7/8 |
| 同一次 session 内已跑过 1+3 | **Step R** → Step 6a → 6b/8 |
| 改了 Step 1 函数 | Step 1 → **Step R** → 目标 Step |
| 改了模型结构 Step 3 | Step 3 → 4（重训）或 Step 3 → **R** |

> **如何同步代码**：在 Colab 用本地最新版 **`260523-74条件生成与量化评估（MVP）.ipynb` 整文件上传/覆盖**，不要只复制某一个 cell 片段。


In [ ]:
# Step R: 快速恢复环境 + 加载权重（~10 秒）
# Colab 重启后请先跑 Step 1 → Step 3，再执行本 cell

import os, json, glob
from datetime import datetime
import torch

if 'device' not in globals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'DATA_DIR' not in globals():
    USE_DRIVE = True
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
        except ImportError:
            USE_DRIVE = False

    def resolve_data_dir():
        candidates = [
            '/content/drive/MyDrive/master_thesis/data',
            '/content/drive/MyDrive/260509_dataset',
            '/content/data',
            os.path.join(os.getcwd(), 'data'),
            os.path.abspath(os.path.join(os.getcwd(), '..', 'data')),
        ]
        for path in candidates:
            if os.path.isdir(path):
                return path
        return candidates[0]

    DATA_DIR = resolve_data_dir()
    OUT_DIR = os.path.join(DATA_DIR, 'processed_tensors_v2')

if 'set_experiment_meta' not in globals():
    TOTAL_JSON_COUNT = len([
        f for f in glob.glob(os.path.join(DATA_DIR, 'house_*.json'))
        if not f.endswith('_topology.json')
    ])
    TRAIN_SAMPLE_COUNT = None
    TRAIN_TIMESTAMP = None
    EXPERIMENT_META = {}

    def set_experiment_meta(train_count, total_count=None, timestamp=None, extra=None):
        global TRAIN_SAMPLE_COUNT, TRAIN_TIMESTAMP, EXPERIMENT_META, TOTAL_JSON_COUNT
        TRAIN_SAMPLE_COUNT = int(train_count)
        if total_count is not None:
            TOTAL_JSON_COUNT = int(total_count)
        TRAIN_TIMESTAMP = timestamp or datetime.now().strftime('%Y%m%d_%H%M%S')
        EXPERIMENT_META = {
            'train_samples': TRAIN_SAMPLE_COUNT,
            'total_json': TOTAL_JSON_COUNT,
            'trained_at': TRAIN_TIMESTAMP,
        }
        if extra:
            EXPERIMENT_META.update(extra)
        return EXPERIMENT_META

_restore_deps = {
    'SpatialModalCVAE': 'Step 3（模型定义）',
    'prepare_graph_batch': 'Step 1（工具函数）',
    'forward_model': 'Step 1（工具函数）',
}
_missing = [f'{name} ← 请先运行 {step}' for name, step in _restore_deps.items() if name not in globals()]
if _missing:
    raise RuntimeError(
        'Colab 重启后内存已清空，不能只跑 Step R。\n'
        '请依次执行：Step 0（如需）→ Step 1 → Step 3 → 再回来执行 Step R\n'
        '当前缺少：\n  - ' + '\n  - '.join(_missing)
    )

WEIGHT_PATH = os.path.join(DATA_DIR, 'spatial_modal_cvae_mvp.pth')

def quick_restore(force_rebuild_model=True):
    global model
    if force_rebuild_model or 'model' not in globals():
        model = SpatialModalCVAE().to(device)
    if os.path.exists(WEIGHT_PATH):
        model.load_state_dict(torch.load(WEIGHT_PATH, map_location=device, weights_only=True))
        print(f'已加载权重: {WEIGHT_PATH}')
    else:
        print(f'警告: 未找到权重 {WEIGHT_PATH}，请先运行 Step 4 训练')
    model.eval()

    meta_files = sorted(glob.glob(os.path.join(DATA_DIR, 'eval_reports', 'experiment_meta_*.json')))
    if meta_files:
        with open(meta_files[-1], 'r', encoding='utf-8') as f:
            meta = json.load(f)
        set_experiment_meta(meta.get('train_samples', 0), meta.get('total_json'), meta.get('trained_at'), meta)
        print('实验元数据:', EXPERIMENT_META)
    print(f'设备: {device} | 数据: {DATA_DIR} | JSON 数: {TOTAL_JSON_COUNT}')
    return model

quick_restore()


In [ ]:

# Step 4: 训练（train/val + 早停 + BCE + Dice + KL）
set_seed(42)

pt_files = sorted(glob.glob(os.path.join(OUT_DIR, '*.pt')))
pt_files = [p for p in pt_files if not os.path.basename(p).startswith('_')]
if not pt_files:
    raise RuntimeError('请先运行 Step 2 预处理')

dataset_list = []
for fp in pt_files:
    try:
        sample = torch.load(fp, weights_only=False)
        hg = sample['graph']
        hg.y = sample['voxel']
        hg.condition = sample['condition'].unsqueeze(0)
        dataset_list.append(hg)
    except Exception as exc:
        print(f'读取失败 {fp}: {exc}')

random.shuffle(dataset_list)
split = max(1, int(len(dataset_list) * 0.8))
train_set, val_set = dataset_list[:split], dataset_list[split:]
print(f'样本总数 {len(dataset_list)} | 训练 {len(train_set)} | 验证 {len(val_set)}')

BATCH_SIZE = 2
train_loader = GraphDataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = GraphDataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False) if val_set else None

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)

EPOCHS = 120
PATIENCE = 20
KL_WEIGHT = 1e-4
DICE_WEIGHT = 0.3

best_val = float('inf')
bad_epochs = 0
history = {'train': [], 'val': []}
import time as _time
_train_start = _time.time()
save_path = os.path.join(DATA_DIR, 'spatial_modal_cvae_mvp.pth')

def run_epoch(loader, train_mode=True):
    model.train(train_mode)
    total = 0.0
    n = 0
    for batch in loader:
        batch = batch.to(device)
        bs = graph_batch_size(batch)
        target = batch.y.view(bs, NUM_CHANNELS, RES_X, RES_Y, RES_Z)

        if train_mode:
            optimizer.zero_grad()

        logits, mu, logvar = forward_model(model, batch)
        bce = F.binary_cross_entropy_with_logits(logits, target)
        dice = dice_loss_with_logits(logits, target)
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        loss = bce + DICE_WEIGHT * dice + KL_WEIGHT * kl

        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total += loss.item() * bs
        n += bs
    return total / max(n, 1)

print(f'开始训练 ({device}) ...')
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, True)
    history['train'].append(tr)
    va = run_epoch(val_loader, False) if val_loader else tr
    history['val'].append(va)
    scheduler.step(va)

    if va < best_val:
        best_val = va
        bad_epochs = 0
        torch.save(model.state_dict(), save_path)
    else:
        bad_epochs += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f'Epoch {epoch:03d}/{EPOCHS} | train={tr:.4f} val={va:.4f} best={best_val:.4f}')

    if bad_epochs >= PATIENCE:
        print(f'早停于 epoch {epoch}，最佳 val={best_val:.4f}')
        break

set_experiment_meta(
    train_count=len(train_set),
    total_count=len(dataset_list),
    extra={
        'val_samples': len(val_set),
        'epochs_run': epoch,
        'best_val_loss': float(best_val),
        'train_seconds': round(_time.time() - _train_start, 1),
    },
)
meta_path = experiment_report_path('experiment_meta', ext='json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(EXPERIMENT_META, f, indent=2, ensure_ascii=False)
print(f'最佳权重已保存: {save_path}')
print(f'实验元数据: {meta_path}')
print(EXPERIMENT_META)

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(history['train'], label='train')
plt.plot(history['val'], label='val')
plt.title('Spatial Modal CVAE Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()




In [ ]:

# Step 5: 重建基线 — GT 异构图 → 体素（性能上界）
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'

COLOR_DICT = {
    1: '#808080', 2: '#FF8000', 3: '#FFFF00', 4: '#00FF00', 5: '#0000FF',
    6: '#FF0000', 7: '#B0B0FF', 8: '#A000FF', 9: '#3CB371', 10: '#00FFFF', 11: '#FFC0CB',
}
NAME_MAP = {
    1: '玄关', 2: '客厅', 3: '餐厅', 4: '厨房', 5: '卧室', 6: '卫生间',
    7: '过道', 8: '楼梯', 9: '储藏', 10: '阳台', 11: '多功能',
}

weight_path = os.path.join(DATA_DIR, 'spatial_modal_cvae_mvp.pth')
model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
model.eval()

sample_pt = random.choice(pt_files)
sample = torch.load(sample_pt, weights_only=False)
hg = sample['graph']
hg.y = sample['voxel']
hg.condition = sample['condition'].unsqueeze(0)
batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)

with torch.no_grad():
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred = torch.argmax(logits[0], dim=0).cpu().numpy()
    gt = torch.argmax(sample['voxel'], dim=0).numpy()

def voxel_coords(class_map, max_pts=8000):
    coords = np.argwhere(class_map > 0)
    if len(coords) > max_pts:
        coords = coords[np.random.choice(len(coords), max_pts, replace=False)]
    xs, ys, zs, cols, texts = [], [], [], [], []
    for ix, iy, iz in coords:
        cid = int(class_map[ix, iy, iz])
        xs.append(ix * VOXEL_SIZE)
        ys.append(iy * VOXEL_SIZE)
        zs.append(iz * VOXEL_SIZE)
        cols.append(COLOR_DICT.get(cid, '#CCC'))
        texts.append(NAME_MAP.get(cid, str(cid)))
    return xs, ys, zs, cols, texts

from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]],
                    subplot_titles=('Ground Truth', 'CVAE Prediction'))

for col, arr in ((1, gt), (2, pred)):
    xs, ys, zs, cols, texts = voxel_coords(arr)
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode='markers',
        marker=dict(size=3, color=cols, symbol='square', opacity=0.85),
        text=texts, hoverinfo='text', showlegend=False,
    ), row=1, col=col)

acc = (pred == gt).mean()
occupied_acc = (pred[gt > 0] == gt[gt > 0]).mean() if (gt > 0).any() else 0.0
print(f'样本: {os.path.basename(sample_pt)}')
print(f'全网格准确率: {acc*100:.2f}% | 非空体素准确率: {occupied_acc*100:.2f}%')

fig.update_layout(height=650, width=1200, title='空间模态 CVAE 体素重建对比')
fig.update_scenes(aspectmode='data')
fig.show()




## Step 6: 条件生成管线

从 **用地尺寸 + 功能清单** 构建合成异构图与条件向量，调用训练好的 CVAE 生成体素，再提取并规整为 300mm 模数体块。


In [ ]:
import plotly.graph_objects as go
# Step 6a: 条件生成核心函数
import math
import networkx as nx

DEFAULT_ROOM_SIZE = {
    'entryway': (2400, 2400, 3000),
    'living_room': (6000, 4500, 3000),
    'dining_room': (3600, 3300, 3000),
    'kitchen': (3300, 3000, 3000),
    'bedroom': (3600, 3600, 3000),
    'bathroom': (2400, 2400, 3000),
    'corridor': (1800, 2400, 3000),
    'stairs': (3000, 3000, 6000),
    'utility': (2400, 2400, 3000),
    'balcony': (3000, 1800, 3000),
    'multi_purpose': (3600, 3300, 3000),
}

TYPE_COLOR_DICT = {
    'entryway': '#808080', 'living_room': '#FF8000', 'dining_room': '#FFFF00',
    'kitchen': '#00FF00', 'bedroom': '#0000FF', 'bathroom': '#FF0000',
    'corridor': '#B0B0FF', 'stairs': '#A000FF', 'utility': '#3CB371',
    'balcony': '#00FFFF', 'multi_purpose': '#FFC0CB',
}

CN_NAMES = {
    'entryway': '玄关', 'living_room': '客厅', 'dining_room': '餐厅', 'kitchen': '厨房',
    'bedroom': '卧室', 'bathroom': '卫生间', 'corridor': '过道', 'stairs': '楼梯',
    'utility': '储藏', 'balcony': '阳台', 'multi_purpose': '多功能',
}


def snap_modulus(v):
    return round(float(v) / VOXEL_SIZE) * VOXEL_SIZE


def build_user_request(site_x, site_y, room_counts, site_z=6000):
    counts = {k: int(v) for k, v in room_counts.items() if int(v) > 0}
    return {
        'site_x': float(site_x), 'site_y': float(site_y), 'site_z': float(site_z),
        'room_counts': counts,
    }


def build_program_topology(room_counts, seed=42):
    G = nx.Graph()
    nodes = []
    bath_i = corr_i = 0
    for r_type, count in room_counts.items():
        for i in range(count):
            nid = f"{r_type}_{i}"
            if r_type in ['entryway', 'living_room', 'dining_room', 'kitchen']:
                floor = 1
            elif r_type in ['bedroom', 'balcony']:
                floor = 2
            elif r_type == 'bathroom':
                floor = 1 if bath_i % 2 == 0 else 2
                bath_i += 1
            elif r_type == 'corridor':
                floor = 1 if corr_i % 2 == 0 else 2
                corr_i += 1
            elif r_type == 'stairs':
                floor = '1&2'
            else:
                floor = 1
            nodes.append((nid, r_type, floor))
            G.add_node(nid, type=r_type, floor=floor)

    f1_corr = [n for n, t, f in nodes if t == 'corridor' and f == 1]
    f2_corr = [n for n, t, f in nodes if t == 'corridor' and f == 2]
    f1_hub = f1_corr[0] if f1_corr else next((n for n, t, f in nodes if t == 'living_room'), nodes[0][0])
    f2_hub = f2_corr[0] if f2_corr else next((n for n, t, f in nodes if t == 'bedroom'), nodes[-1][0])

    edge_types = {}
    for nid, r_type, floor in nodes:
        if r_type == 'stairs':
            continue
        hub = f1_hub if floor == 1 else f2_hub
        if nid != hub:
            G.add_edge(nid, hub)
            edge_types[(nid, hub)] = 'horizontal'
            edge_types[(hub, nid)] = 'horizontal'

    stairs = [n for n, t, f in nodes if t == 'stairs']
    if stairs:
        st = stairs[0]
        if f1_hub != st:
            G.add_edge(st, f1_hub)
            edge_types[(st, f1_hub)] = 'vertical'
        if f2_hub != st and f2_hub != f1_hub:
            G.add_edge(st, f2_hub)
            edge_types[(st, f2_hub)] = 'vertical'

    pos = nx.spring_layout(G, seed=seed, k=1.2)
    return G, pos, nodes, edge_types


def layout_rooms_from_program(user_req, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    G, pos, nodes, edge_types = build_program_topology(user_req['room_counts'], seed=seed)
    sx, sy = user_req['site_x'], user_req['site_y']
    rooms = []
    for nid, r_type, floor in nodes:
        w, d, h = DEFAULT_ROOM_SIZE.get(r_type, (3600, 3600, 3000))
        px, py = pos[nid]
        cx = (px + 1) / 2 * (sx * 0.7) + sx * 0.15
        cy = (py + 1) / 2 * (sy * 0.7) + sy * 0.15
        cx, cy = snap_modulus(cx), snap_modulus(cy)
        w, d = snap_modulus(w), snap_modulus(d)
        z0 = 0 if floor == 1 or floor == '1&2' else 3000
        z1 = z0 + h
        rooms.append({
            'id': nid, 'type': r_type, 'floor': 1 if floor == '1&2' else floor,
            'box_min': [max(0, cx - w / 2), max(0, cy - d / 2), z0],
            'box_max': [min(sx, cx + w / 2), min(sy, cy + d / 2), z1],
            'lighting_access': 'direct' if r_type in ['living_room', 'bedroom', 'balcony'] else 'indirect',
            'lighting_priority': 8 if r_type in ['living_room', 'bedroom'] else 4,
            'effective_lighting': [],
        })
    return rooms, G, pos, edge_types


def request_to_house_json(user_req, rooms):
    stats = {t: 0 for t in ROOM_TYPES}
    shown_types = set()
    for r in rooms:
        stats[r['type']] = stats.get(r['type'], 0) + 1
    return {
        'metadata': {
            'stats': stats,
            'total_rooms': len(rooms),
            'building_size': {'x': user_req['site_x'], 'y': user_req['site_y'], 'z': user_req['site_z']},
            'constraints': {'modulus': int(VOXEL_SIZE)},
        },
        'rooms': rooms,
    }


@torch.no_grad()
def generate_voxels_from_request(user_req, model, seed=42):
    rooms, G, pos, edge_types = layout_rooms_from_program(user_req, seed=seed)
    data = request_to_house_json(user_req, rooms)
    sample = json_to_sample(data)
    hg = sample['graph']
    batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)
    model.eval()
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred_cls = torch.argmax(logits[0], dim=0).cpu().numpy()
    return pred_cls, sample, rooms, G, pos


def count_components_3d(mask):
    if not mask.any():
        return 0
    visited = np.zeros(mask.shape, dtype=bool)
    n_comp = 0
    xs, ys, zs = np.where(mask)
    for x, y, z in zip(xs, ys, zs):
        if visited[x, y, z]:
            continue
        n_comp += 1
        stack = [(int(x), int(y), int(z))]
        visited[x, y, z] = True
        while stack:
            cx, cy, cz = stack.pop()
            for dx, dy, dz in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
                nx, ny, nz = cx+dx, cy+dy, cz+dz
                if 0 <= nx < mask.shape[0] and 0 <= ny < mask.shape[1] and 0 <= nz < mask.shape[2]:
                    if mask[nx, ny, nz] and not visited[nx, ny, nz]:
                        visited[nx, ny, nz] = True
                        stack.append((nx, ny, nz))
    return n_comp


def voxels_to_boxes(pred_cls, user_req):
    TYPE_NAMES_LOCAL = {v: k for k, v in CHANNEL_MAP.items() if v > 0}
    boxes = []
    for cid in range(1, NUM_CHANNELS):
        mask = pred_cls == cid
        if not mask.any():
            continue
        visited = np.zeros(mask.shape, dtype=bool)
        xs, ys, zs = np.where(mask)
        for x, y, z in zip(xs, ys, zs):
            if visited[x, y, z]:
                continue
            stack = [(int(x), int(y), int(z))]
            comp = []
            visited[x, y, z] = True
            while stack:
                cx, cy, cz = stack.pop()
                comp.append((cx, cy, cz))
                for dx, dy, dz in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
                    nx, ny, nz = cx+dx, cy+dy, cz+dz
                    if 0 <= nx < mask.shape[0] and 0 <= ny < mask.shape[1] and 0 <= nz < mask.shape[2]:
                        if mask[nx, ny, nz] and not visited[nx, ny, nz]:
                            visited[nx, ny, nz] = True
                            stack.append((nx, ny, nz))
            arr = np.array(comp)
            ix0, iy0, iz0 = arr.min(axis=0)
            ix1, iy1, iz1 = arr.max(axis=0) + 1
            rtype = TYPE_NAMES_LOCAL[cid]
            boxes.append({
                'type': rtype,
                'box_min': [snap_modulus(ix0*VOXEL_SIZE), snap_modulus(iy0*VOXEL_SIZE), snap_modulus(iz0*VOXEL_SIZE)],
                'box_max': [snap_modulus(ix1*VOXEL_SIZE), snap_modulus(iy1*VOXEL_SIZE), snap_modulus(iz1*VOXEL_SIZE)],
            })
    return boxes






def rooms_to_boxes(rooms):
    return [{'type': r['type'], 'box_min': list(r['box_min']), 'box_max': list(r['box_max'])} for r in rooms]


def plot_3d_layout(rooms, site_x=None, site_y=None, title='AI 生成的 3D 功能体块布局图'):
    """与 260510 gemini 版一致：Mesh3d 半透明 + Scatter3d 边框"""
    fig = go.Figure()
    shown_types = set()
    for r in rooms:
        x0, y0, z0 = r['box_min']
        x1, y1, z1 = r['box_max']
        if x1 <= x0 or y1 <= y0 or z1 <= z0:
            continue
        color = TYPE_COLOR_DICT.get(r['type'], '#CCCCCC')
        x_edges = [x0, x1, x1, x0, x0, x0, x1, x1, x0, x0, x1, x1, x1, x1, x0, x0]
        y_edges = [y0, y0, y1, y1, y0, y0, y0, y1, y1, y0, y0, y0, y1, y1, y1, y1]
        z_edges = [z0, z0, z0, z0, z0, z1, z1, z1, z1, z1, z1, z0, z0, z1, z1, z0]
        fig.add_trace(go.Scatter3d(
            x=x_edges, y=y_edges, z=z_edges, mode='lines',
            line=dict(color=color, width=5),
            name=f"{CN_NAMES.get(r['type'], r['type'])} ({r.get('id', r['type'])})",
            showlegend=(r['type'] not in shown_types),
            legendgroup=r['type'],
        ))
        fig.add_trace(go.Mesh3d(
            x=[x0, x0, x1, x1, x0, x0, x1, x1],
            y=[y0, y1, y1, y0, y0, y1, y1, y0],
            z=[z0, z0, z0, z0, z1, z1, z1, z1],
            i=[7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2],
            j=[3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3],
            k=[0, 7, 2, 3, 6, 7, 1, 1, 5, 5, 7, 6],
            opacity=0.25, color=color, hoverinfo='text',
            text=f"{CN_NAMES.get(r['type'], r['type'])}<br>{x1-x0:.0f}x{y1-y0:.0f}x{z1-z0:.0f} mm",
            showlegend=False,
            legendgroup=r['type'],
        ))
        shown_types.add(r['type'])
    scene = dict(xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)', aspectmode='data')
    if site_x and site_y:
        scene['xaxis'] = dict(title='X (mm)', range=[0, site_x])
        scene['yaxis'] = dict(title='Y (mm)', range=[0, site_y])
        scene['zaxis'] = dict(title='Z (mm)', range=[0, 9000])
    fig.update_layout(title=title, width=1000, height=720, scene=scene,
                      margin=dict(l=0, r=0, b=0, t=50))
    return fig


def show_plotly_fig(fig):
    """Colab 可靠显示 Plotly 3D。ipywidgets.Output 内调用 fig.show() 常会空白。"""
    from IPython.display import HTML, display
    html = fig.to_html(
        full_html=False, include_plotlyjs='cdn',
        config={'displayModeBar': True, 'responsive': True},
    )
    display(HTML(html))


@torch.no_grad()
def infer_voxels_multi_strategy(user_req, rooms, model):
    """尝试多种解码策略，缓解合成拓扑下体素全空的问题"""
    data = request_to_house_json(user_req, rooms)
    sample = json_to_sample(data)
    hg = sample['graph']
    cond = sample['condition']
    batch = prepare_graph_batch(hg, condition=cond).to(device)
    cond_dev = cond.unsqueeze(0).to(device) if cond.dim() == 1 else cond.to(device)
    model.eval()

    candidates = []
    logits, mu, logvar = forward_model(model, batch, cond)
    pred_graph = torch.argmax(logits[0], dim=0).cpu().numpy()
    candidates.append(('graph_forward', pred_graph, int((pred_graph > 0).sum())))

    logits_mu = model.decoder(mu, cond_dev)
    pred_mu = torch.argmax(logits_mu[0], dim=0).cpu().numpy()
    candidates.append(('encoder_mu', pred_mu, int((pred_mu > 0).sum())))

    z_rand = torch.randn(1, LATENT_DIM, device=device)
    logits_rand = model.decoder(z_rand, cond_dev)
    pred_rand = torch.argmax(logits_rand[0], dim=0).cpu().numpy()
    candidates.append(('random_z_cond', pred_rand, int((pred_rand > 0).sum())))

    best = max(candidates, key=lambda x: x[2])
    return best[1], best[2], best[0], candidates


def generate_user_layout(user_req, model, seed=None):
    if seed is None:
        seed = random.randint(1, 999999)
    rooms, G, pos, edge_types = layout_rooms_from_program(user_req, seed=seed)
    pred, n_occ, mode, _ = infer_voxels_multi_strategy(user_req, rooms, model)
    if n_occ > 0:
        display_rooms = voxels_to_boxes(pred)
        display_source = 'model_voxels'
    else:
        display_rooms = rooms
        display_source = 'layout_fallback'
    return {
        'rooms': rooms, 'display_rooms': display_rooms, 'display_source': display_source,
        'graph': G, 'pos': pos, 'pred': pred,
        'n_occ': n_occ, 'decode_mode': mode, 'seed': seed,
    }


print('Step 6a 就绪: plot_3d_layout / generate_user_layout')





In [ ]:
# Step 6b: 单次条件生成 + Colab 原生 3D 预览（fig.show）
import plotly.io as pio
pio.renderers.default = 'colab'

quick_restore(force_rebuild_model=False)

demo_req = build_user_request(
    site_x=18000, site_y=15000,
    room_counts={
        'entryway': 1, 'living_room': 1, 'dining_room': 1, 'kitchen': 1,
        'bedroom': 3, 'bathroom': 2, 'corridor': 2, 'stairs': 1, 'balcony': 1,
    },
)
result = generate_user_layout(demo_req, model, seed=123)
print('seed:', result['seed'])
print('解码策略:', result['decode_mode'], '| 非空体素:', result['n_occ'])
print('布局体块数:', len(result['rooms']))
fig = plot_3d_layout(result['rooms'], 18000, 15000, title='条件生成三维功能体块（Colab 预览）')
fig.show()


## Step 7: 量化评估（重建 vs 条件生成）

- **重建模式**：GT 异构图（上界）
- **生成模式**：仅用户约束 → 合成拓扑（真实使用路径）

导出命名格式：`{报告名}_json{总数}_train{训练数}_{时间}.csv/json`
例如：`generation_metrics_json74_train59_20260523_143022.csv`


In [ ]:
# Step 7a: 量化指标函数
EVAL_GEN_DIR = os.path.join(DATA_DIR, 'eval_reports')
os.makedirs(EVAL_GEN_DIR, exist_ok=True)


def count_rooms_by_type(boxes):
    counts = {t: 0 for t in ROOM_TYPES}
    for b in boxes:
        counts[b['type']] = counts.get(b['type'], 0) + 1
    return counts


def program_count_mae(requested, predicted):
    keys = set(requested.keys()) | set(predicted.keys())
    diffs = [abs(int(requested.get(k, 0)) - int(predicted.get(k, 0))) for k in keys]
    return float(np.mean(diffs)) if diffs else 0.0


def program_count_acc(requested, predicted):
    keys = set(requested.keys()) | set(predicted.keys())
    ok, total = 0, 0
    for k in keys:
        total += max(int(requested.get(k, 0)), 1)
        ok += min(int(requested.get(k, 0)), int(predicted.get(k, 0)))
    return float(ok / max(total, 1))


def modulus_compliance(boxes):
    if not boxes:
        return 0.0
    ok = 0
    total = 0
    for b in boxes:
        for k in ('box_min', 'box_max'):
            for v in b[k]:
                total += 1
                if abs(v / VOXEL_SIZE - round(v / VOXEL_SIZE)) < 1e-6:
                    ok += 1
    return ok / max(total, 1)


def site_fit_rate(boxes, site_x, site_y, site_z=9000):
    if not boxes:
        return 0.0
    ok = 0
    for b in boxes:
        bm, bx = b['box_min'], b['box_max']
        if bm[0] >= -1 and bm[1] >= -1 and bx[0] <= site_x + 1 and bx[1] <= site_y + 1 and bx[2] <= site_z + 1:
            ok += 1
    return ok / len(boxes)


def voxel_metrics_np(pred_cls, gt_cls):
    pred_cls = pred_cls.astype(np.int16)
    gt_cls = gt_cls.astype(np.int16)
    total_acc = float((pred_cls == gt_cls).mean())
    occ = gt_cls > 0
    occupied_acc = float((pred_cls[occ] == gt_cls[occ]).mean()) if occ.any() else 0.0
    ious = []
    for cid in range(1, NUM_CHANNELS):
        p, g = pred_cls == cid, gt_cls == cid
        if not g.any():
            continue
        inter = np.logical_and(p, g).sum()
        union = np.logical_or(p, g).sum()
        ious.append(inter / max(union, 1))
    return total_acc, occupied_acc, float(np.mean(ious) if ious else 0.0)


def json_to_user_request(data):
    meta = data.get('metadata', {})
    stats = meta.get('stats', {})
    bsize = meta.get('building_size', {})
    counts = {t: int(stats.get(t, 0)) for t in ROOM_TYPES if int(stats.get(t, 0)) > 0}
    return build_user_request(
        site_x=float(bsize.get('x', 18000)),
        site_y=float(bsize.get('y', 15000)),
        site_z=float(bsize.get('z', 6000)),
        room_counts=counts,
    )


@torch.no_grad()
def eval_reconstruction(sample, model):
    hg = sample['graph']
    hg.condition = sample['condition'].unsqueeze(0) if sample['condition'].dim() == 1 else sample['condition']
    batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred = torch.argmax(logits[0], dim=0).cpu().numpy()
    gt = torch.argmax(sample['voxel'], dim=0).numpy()
    ta, oa, miou = voxel_metrics_np(pred, gt)
    return {'total_acc': ta, 'occupied_acc': oa, 'miou': miou, 'pred': pred, 'gt': gt}


@torch.no_grad()
def eval_conditional_generation(user_req, gt_voxel, model, seed=42):
    pred, sample, rooms, G, pos = generate_voxels_from_request(user_req, model, seed=seed)
    gt = torch.argmax(gt_voxel, dim=0).numpy()
    ta, oa, miou = voxel_metrics_np(pred, gt)
    boxes = voxels_to_boxes(pred, user_req)
    pred_counts = count_rooms_by_type(boxes)
    return {
        'total_acc': ta, 'occupied_acc': oa, 'miou': miou,
        'program_mae': program_count_mae(user_req['room_counts'], pred_counts),
        'program_acc': program_count_acc(user_req['room_counts'], pred_counts),
        'modulus_rate': modulus_compliance(boxes),
        'site_fit': site_fit_rate(boxes, user_req['site_x'], user_req['site_y'], user_req['site_z']),
        'fragmentation': float(np.mean([count_components_3d(pred == cid) for cid in range(1, NUM_CHANNELS) if (pred == cid).any()]) if (pred > 0).any() else 0.0),
        'pred_counts': pred_counts,
        'boxes': boxes,
        'pred': pred,
        'gt': gt,
    }

print('Step 7a 就绪')



In [ ]:
# Step 7b: 验证集批量量化评估（重建 vs 条件生成）
set_seed(42)
weight_path = os.path.join(DATA_DIR, 'spatial_modal_cvae_mvp.pth')
model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
model.eval()

pt_files = sorted([
    p for p in glob.glob(os.path.join(OUT_DIR, '*.pt'))
    if not os.path.basename(p).startswith('_')
])
if not pt_files:
    raise RuntimeError('请先运行 Step 2 预处理')

items = []
for fp in pt_files:
    s = torch.load(fp, weights_only=False)
    items.append((os.path.basename(fp), s))

random.shuffle(items)
split = max(1, int(len(items) * 0.8))
val_items = items[split:]
print(f'验证样本数: {len(val_items)}')

rows = []
for pt_name, sample in val_items:
    json_path = os.path.join(DATA_DIR, pt_name.replace('.pt', '.json'))
    if not os.path.exists(json_path):
        print('跳过，无 JSON:', pt_name)
        continue
    with open(json_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    user_req = json_to_user_request(raw)

    hg = sample['graph']
    hg.condition = sample['condition'].unsqueeze(0) if sample['condition'].dim() == 1 else sample['condition']
    rec_sample = {'graph': hg, 'voxel': sample['voxel'], 'condition': sample['condition']}

    rec = eval_reconstruction(rec_sample, model)
    gen = eval_conditional_generation(user_req, sample['voxel'], model, seed=42)

    rows.append({
        'sample': pt_name.replace('.pt', ''),
        'mode': 'reconstruction',
        'miou': rec['miou'], 'occupied_acc': rec['occupied_acc'], 'total_acc': rec['total_acc'],
        'program_mae': np.nan, 'program_acc': np.nan, 'modulus_rate': np.nan, 'site_fit': np.nan,
    })
    rows.append({
        'sample': pt_name.replace('.pt', ''),
        'mode': 'conditional_generation',
        'miou': gen['miou'], 'occupied_acc': gen['occupied_acc'], 'total_acc': gen['total_acc'],
        'program_mae': gen['program_mae'], 'program_acc': gen['program_acc'],
        'modulus_rate': gen['modulus_rate'], 'site_fit': gen['site_fit'],
    })

import pandas as pd
df_gen = pd.DataFrame(rows)

print('\n=== 重建 vs 条件生成 对比 ===')
for mode in ['reconstruction', 'conditional_generation']:
    sub = df_gen[df_gen['mode'] == mode]
    if sub.empty:
        continue
    print(f"\n[{mode}] n={len(sub)}")
    print(f"  mIoU:          {sub['miou'].mean()*100:5.2f}% ± {sub['miou'].std()*100:4.2f}%")
    print(f"  非空准确率:    {sub['occupied_acc'].mean()*100:5.2f}% ± {sub['occupied_acc'].std()*100:4.2f}%")
    if mode == 'conditional_generation':
        print(f"  功能数量MAE:   {sub['program_mae'].mean():5.2f} ± {sub['program_mae'].std():4.2f}")
        print(f"  功能数量匹配率:{sub['program_acc'].mean()*100:5.2f}% ± {sub['program_acc'].std()*100:4.2f}%")
        print(f"  模数合规率:    {sub['modulus_rate'].mean()*100:5.2f}%")
        print(f"  用地贴合率:    {sub['site_fit'].mean()*100:5.2f}%")

csv_out = experiment_report_path('generation_metrics', subdir='eval_reports', ext='csv')
json_out = experiment_report_path('generation_summary', subdir='eval_reports', ext='json')
df_gen.to_csv(csv_out, index=False, encoding='utf-8-sig')

report = {}
for mode in df_gen['mode'].unique():
    sub = df_gen[df_gen['mode'] == mode]
    report[mode] = {
        c: float(sub[c].mean()) for c in [
            'miou', 'occupied_acc', 'total_acc', 'program_mae', 'program_acc', 'modulus_rate', 'site_fit'
        ] if c in sub.columns and sub[c].notna().any()
    }
with open(json_out, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\n已导出: {csv_out}")
print(f"已导出: {json_out}")
try:
    display(df_gen.pivot_table(index='sample', columns='mode', values=['miou', 'occupied_acc', 'program_acc']))
except NameError:
    print(df_gen.head(10))




## Step 8: 交互式条件生成（Colab 推荐：ipywidgets）

- **3D 显示**：用 `show_plotly_fig()`（HTML 嵌入），**不要**在 `widgets.Output` 里直接 `fig.show()`，否则 Colab 经常空白。
- **`非空体素=0`**：模型对「合成拓扑」解码失败，界面会 **自动用布局引擎体块兜底**（仍应能看到 3D）。
- **Gradio**：Colab 内 3D WebGL 不稳定，**仅对外分享时用**；日常请用本 Step。


In [ ]:
# Step 8: Colab 交互生成（ipywidgets + Plotly 3D，推荐）
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.io as pio
pio.renderers.default = 'colab'

quick_restore(force_rebuild_model=False)

w_site_x = widgets.IntSlider(value=18000, min=6000, max=30000, step=300, description='用地X', style={'description_width': '60px'}, layout=widgets.Layout(width='420px'))
w_site_y = widgets.IntSlider(value=15000, min=6000, max=30000, step=300, description='用地Y', style={'description_width': '60px'}, layout=widgets.Layout(width='420px'))
w_seed = widgets.IntText(value=0, description='种子(0=随机)', style={'description_width': '80px'})

fields = [
    ('entryway', '玄关', 1), ('living_room', '客厅', 1), ('dining_room', '餐厅', 1), ('kitchen', '厨房', 1),
    ('bedroom', '卧室', 3), ('bathroom', '卫生间', 2), ('corridor', '过道', 2),
    ('stairs', '楼梯', 1), ('utility', '储藏', 0), ('balcony', '阳台', 1), ('multi_purpose', '多功能', 0),
]
room_widgets = {}
rows = []
for i in range(0, len(fields), 4):
    row = []
    for key, label, default in fields[i:i+4]:
        w = widgets.IntText(value=default, description=label, style={'description_width': '50px'}, layout=widgets.Layout(width='140px'))
        room_widgets[key] = w
        row.append(w)
    rows.append(widgets.HBox(row))

btn = widgets.Button(description='生成三维体块', button_style='primary', icon='cube')
text_out = widgets.Output()
plot_out = widgets.Output(layout=widgets.Layout(min_height='760px', border='1px solid #ddd'))


def on_generate(_=None):
    with text_out:
        clear_output(wait=True)
        counts = {k: int(w.value) for k, w in room_widgets.items() if int(w.value) > 0}
        req = build_user_request(w_site_x.value, w_site_y.value, counts)
        seed = int(w_seed.value) if int(w_seed.value) > 0 else None
        result = generate_user_layout(req, model, seed=seed)
        plot_rooms = result.get('display_rooms', result['rooms'])
        boxes = rooms_to_boxes(plot_rooms)
        pred_counts = count_rooms_by_type(boxes)
        src = '模型体素' if result.get('display_source') == 'model_voxels' else '布局引擎兜底'

        print(f"seed={result['seed']} | 解码={result['decode_mode']} | 非空体素={result['n_occ']} | 显示={src}")
        if result['n_occ'] == 0:
            print('提示: 模型体素为空（合成拓扑与训练数据分布不一致），已用布局引擎体块显示。可固定种子 123 多试几次。')
        print(f"请求: {req['room_counts']}")
        print(f"布局: {pred_counts}")
        print(f"功能MAE={program_count_mae(req['room_counts'], pred_counts):.2f} | "
              f"匹配率={program_count_acc(req['room_counts'], pred_counts)*100:.1f}% | "
              f"模数={modulus_compliance(boxes)*100:.1f}% | 用地={site_fit_rate(boxes, req['site_x'], req['site_y'])*100:.1f}%")

    with plot_out:
        clear_output(wait=True)
        fig = plot_3d_layout(
            plot_rooms, req['site_x'], req['site_y'],
            title=f"条件生成 3D 功能体块 (seed={result['seed']}, {src})",
        )
        show_plotly_fig(fig)

btn.on_click(on_generate)

display(widgets.VBox([
    widgets.HTML('<b>输入用地与功能需求 → 点击下方按钮生成（可反复点击，输入不会清空）</b>'),
    widgets.HBox([w_site_x, w_site_y, w_seed]),
    *rows,
    btn,
    text_out,
    plot_out,
]))
print('Step 8 就绪：文字在上、3D 在下；若仍空白请先运行 Step 6a 再重跑本 cell')


### Step 8-可选：Gradio 网页 UI（需对外分享时用）

注意：`debug=True` 会导致 Colab **反复重跑单元格**；已改为 `debug=False`。
3D 预览在 Colab 内仍建议用上方 **Step 8 ipywidgets**。


In [ ]:
# Step 8-可选: Gradio（debug=False 避免自动重跑）
# !pip -q install gradio

import gradio as gr

quick_restore(force_rebuild_model=False)


def gradio_generate(site_x, site_y, seed_input, n_entry, n_living, n_dining, n_kitchen,
                    n_bed, n_bath, n_corr, n_stairs, n_util, n_balc, n_multi):
    req = build_user_request(site_x, site_y, {
        'entryway': int(n_entry), 'living_room': int(n_living), 'dining_room': int(n_dining),
        'kitchen': int(n_kitchen), 'bedroom': int(n_bed), 'bathroom': int(n_bath),
        'corridor': int(n_corr), 'stairs': int(n_stairs), 'utility': int(n_util),
        'balcony': int(n_balc), 'multi_purpose': int(n_multi),
    })
    seed = int(seed_input) if int(seed_input) > 0 else None
    out = generate_user_layout(req, model, seed=seed)
    boxes = rooms_to_boxes(out['rooms'])
    fig = plot_3d_layout(out['rooms'], site_x, site_y, title=f'seed={out["seed"]}')
    html = fig.to_html(full_html=False, include_plotlyjs='cdn')
    metrics = (
        f"seed={out['seed']}\n非空体素={out['n_occ']}\n解码={out['decode_mode']}\n"
        f"匹配率={program_count_acc(req['room_counts'], count_rooms_by_type(boxes))*100:.1f}%"
    )
    return html, metrics


with gr.Blocks() as demo:
    gr.Markdown('## Gradio 条件生成（可选）')
    with gr.Row():
        site_x = gr.Slider(6000, 30000, value=18000, step=300, label='X')
        site_y = gr.Slider(6000, 30000, value=15000, step=300, label='Y')
        seed_in = gr.Number(0, label='种子', precision=0)
    with gr.Row():
        n_entry = gr.Number(1, label='玄关', precision=0)
        n_living = gr.Number(1, label='客厅', precision=0)
        n_dining = gr.Number(1, label='餐厅', precision=0)
        n_kitchen = gr.Number(1, label='厨房', precision=0)
    with gr.Row():
        n_bed = gr.Number(3, label='卧室', precision=0)
        n_bath = gr.Number(2, label='卫生间', precision=0)
        n_corr = gr.Number(2, label='过道', precision=0)
    with gr.Row():
        n_stairs = gr.Number(1, label='楼梯', precision=0)
        n_util = gr.Number(0, label='储藏', precision=0)
        n_balc = gr.Number(1, label='阳台', precision=0)
        n_multi = gr.Number(0, label='多功能', precision=0)
    btn = gr.Button('生成')
    plot_html = gr.HTML(label='3D HTML')
    metrics = gr.Textbox(label='指标')
    btn.click(gradio_generate,
              inputs=[site_x, site_y, seed_in, n_entry, n_living, n_dining, n_kitchen,
                      n_bed, n_bath, n_corr, n_stairs, n_util, n_balc, n_multi],
              outputs=[plot_html, metrics])

demo.launch(debug=False, share=True)
